# Cell 1: Library Imports

In [0]:
import boto3
import json
import gzip
import io
import time
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql import Row
from pyspark.sql.functions import to_timestamp, current_timestamp, col

# Cell 2: Configuration & Widgets

In [0]:
# Widgets for parameterization
dbutils.widgets.text("s3_bucket", "enterprise-raw-lakehouse")
dbutils.widgets.text("s3_prefix", "macro/fred") 
dbutils.widgets.text("catalog", "enterprise")
dbutils.widgets.text("schema", "bronze_ingestion")
dbutils.widgets.text("table_name", "brz_macro_fred_series")
dbutils.widgets.text("aws_access_key", "", "AWS Access Key")
dbutils.widgets.text("aws_secret_key", "", "AWS Secret Key")
dbutils.widgets.text("aws_region", "eu-central-1", "AWS Region")
dbutils.widgets.text("mode", "batch") # 'batch' or 'realtime'

# Parse parameters
BUCKET = dbutils.widgets.get("s3_bucket").strip()
PREFIX = dbutils.widgets.get("s3_prefix").strip().strip("/")
CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
TABLE_NAME = dbutils.widgets.get("table_name").strip()
ACCESS_KEY = dbutils.widgets.get("aws_access_key").strip()
SECRET_KEY = dbutils.widgets.get("aws_secret_key").strip()
REGION = dbutils.widgets.get("aws_region").strip()
MODE = dbutils.widgets.get("mode").strip().lower()

TARGET_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

print(f"🚀 Source Base: s3://{BUCKET}/{PREFIX}/")
print(f"🎯 Target: {TARGET_TABLE}")
print(f"⚙️ Mode:   {MODE.upper()}")

# Cell 3: Initialize S3 Client

In [0]:
if not ACCESS_KEY or not SECRET_KEY:
    raise ValueError("❌ Missing AWS Keys! Please provide aws_access_key and aws_secret_key.")

s3 = boto3.client(
    "s3",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=REGION
)
print("✅ S3 Client Initialized.")

# Cell 4: Scan S3 files -> Deduplication -> Write to Delta  

In [0]:
def list_jsonl_files(bucket, prefix):
    """List all .jsonl.gz files recursively in the bucket/prefix"""
    files = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if "Contents" in page:
            for obj in page["Contents"]:
                key = obj["Key"]
                if key.endswith(".jsonl.gz"):
                    files.append(key)
    return files

def run_batch():
    start_time = datetime.now()
    print(f"\n--- ⏳ Starting Batch at {start_time} ---")

    # --- Step 1: List Files (Recursive) ---
    print(f"🔍 Listing files recursively in {PREFIX}...")
    all_file_keys = list_jsonl_files(BUCKET, PREFIX)
    print(f"📄 Found total {len(all_file_keys)} files in S3 under {PREFIX}/.")

    # --- Step 2: Deduplication (Checkpointing) ---
    processed_files = set()
    try:
        if spark.catalog.tableExists(TARGET_TABLE):
            df_existing = spark.sql(f"SELECT DISTINCT source_file FROM {TARGET_TABLE} WHERE source_file IS NOT NULL")
            rows = df_existing.collect()
            processed_files = {r["source_file"] for r in rows if r["source_file"]}
            print(f"📚 Found {len(processed_files)} already processed files in Delta table.")
    except Exception as e:
        print(f"⚠️ Warning: Could not query existing table: {e}. Proceeding with all files.")

    # Filter out already processed files
    file_keys = [k for k in all_file_keys if k not in processed_files]
    skipped_count = len(all_file_keys) - len(file_keys)

    if skipped_count > 0:
        print(f"⏭️ Skipped {skipped_count} files (already ingested).")
    
    # --- Step 3: Process in Chunks ---
    files_to_process = file_keys
    CHUNK_SIZE = 500  # Adjust based on memory
    total_files = len(files_to_process)
    
    if total_files == 0:
        print("✅ No new files to process.")
        return

    print(f"🚀 Total files to process: {total_files}. Processing in chunks of {CHUNK_SIZE}...")

    for i in range(0, total_files, CHUNK_SIZE):
        chunk_files = files_to_process[i : i + CHUNK_SIZE]
        all_rows = []
        error_count = 0
        
        print(f"\n--- Batch {i // CHUNK_SIZE + 1} / {(total_files // CHUNK_SIZE) + 1} ---")
        print(f"📥 Reading files {i} to {min(i + CHUNK_SIZE, total_files)}...")

        for key in chunk_files:
            try:
                obj = s3.get_object(Bucket=BUCKET, Key=key)
                with gzip.GzipFile(fileobj=obj["Body"]) as gz:
                    for line in gz:
                        if line.strip():
                            record = json.loads(line)
                            record["source_file"] = key # Add metadata for deduplication
                            all_rows.append(record)
            except Exception as e:
                print(f"❌ Error reading {key}: {e}")
                error_count += 1

        print(f"✅ Extracted {len(all_rows)} records in this chunk. Errors: {error_count}")

        # --- Step 4: Write Chunk to Delta (With Retry) ---
        if all_rows:
            max_retries = 3
            for attempt in range(max_retries):
                try:
                    schema = StructType([
                        StructField("series_id", StringType(), True),
                        StructField("raw_json", StringType(), True),
                        StructField("run_id", StringType(), True),
                        StructField("ingestion_ts", StringType(), True),
                        StructField("source", StringType(), True),
                        StructField("mode", StringType(), True),
                        StructField("source_file", StringType(), True)
                    ])

                    df = spark.createDataFrame(all_rows, schema=schema)
                    
                    # Transformations & Timestamp Casting
                    df_final = (df
                        .withColumn("ingestion_ts", to_timestamp(col("ingestion_ts"))) 
                        .withColumn("bronze_proc_ts", current_timestamp())
                    )

                    # Create Database if not exists
                    spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG}.{SCHEMA}")

                    (df_final.write
                        .format("delta")
                        .mode("append")
                        .option("mergeSchema", "true")
                        .saveAsTable(TARGET_TABLE)
                    )
                    print(f"💾 Written chunk to {TARGET_TABLE}")
                    break # Success! Exit retry loop
                except Exception as e:
                    wait_time = 15 * (attempt + 1)
                    print(f"⚠️ Write failed (Attempt {attempt+1}/{max_retries}): {e}")
                    if attempt < max_retries - 1:
                        print(f"⏳ Sleeping {wait_time}s before retrying...")
                        time.sleep(wait_time)
                    else:
                        print("❌ Max retries reached. Skipping this chunk (or stopping).")
                        raise e
            
            # Clear memory
            del all_rows
            del df
            del df_final
        else:
            print("⚠️ No valid rows extracted in this chunk.")

# Cell 5: Execution Control

In [0]:
if 'boto3' not in globals() or 's3' not in globals():
    print("❌ Error: Setup cells not run! Please run the 'Imports' and 'Configuration' cells above first.")
else:
    if MODE == "realtime":
        print(f"🔁 Entering REALTIME mode ({MODE}). Press Stop to cancel.")
        print("   (Polling every 60 seconds...)")
        while True:
            try:
                run_batch()
                print("💤 Sleeping 60s...")
                time.sleep(60)
            except KeyboardInterrupt:
                print("🛑 Stopped by user.")
                break
            except Exception as e:
                print(f"❌ Error in loop: {e}")
                time.sleep(60)
    else:
        print(f"▶️ Running SINGLE BATCH ({MODE}).")
        run_batch()

# Cell 6: Verify Data in Bronze Table

In [0]:
if spark.catalog.tableExists(TARGET_TABLE):
    print(f"📊 Displaying Summary from {TARGET_TABLE}:")
    display(spark.sql(f"""
        SELECT 
            series_id, 
            COUNT(*) as files_loaded, 
            MAX(ingestion_ts) as last_ingested_ts
        FROM {TARGET_TABLE} 
        GROUP BY series_id
    """))
else:
    print(f"⚠️ Table {TARGET_TABLE} does not exist yet. Run the batch first.")

In [0]:
if spark.catalog.tableExists(TARGET_TABLE):
    print("🔎 Unpacking top 20 observations from the JSON payload...")
    display(spark.sql(f"""
        SELECT 
            series_id,
            -- FRED的数据观测值在一个叫做 'observations' 的JSON数组里，我们用 inline(from_json) 把它炸开
            obs.date as observation_date,
            obs.value as actual_value,
            ingestion_ts
        FROM (
            SELECT 
                series_id, 
                ingestion_ts,
                explode(from_json(raw_json, 'struct<observations:array<struct<date:string,value:string>>>').observations) as obs
            FROM {TARGET_TABLE}
        )
        ORDER BY series_id, observation_date DESC
        LIMIT 20
    """))
else:
    print(f"⚠️ Table {TARGET_TABLE} does not exist yet. Run the batch first.")

In [0]:
if spark.catalog.tableExists(TARGET_TABLE):
    print("🔎 cpi/dff/dxy...")
    display(spark.sql(f"""
        WITH UnpackedData AS (
            SELECT 
                series_id,
                obs.date as observation_date,
                obs.value as actual_value,
                ingestion_ts,
                -- 给每个指标的数据按日期倒序排个号
                ROW_NUMBER() OVER (PARTITION BY series_id ORDER BY obs.date DESC) as row_num
            FROM (
                SELECT 
                    series_id, 
                    ingestion_ts,
                    explode(from_json(raw_json, 'struct<observations:array<struct<date:string,value:string>>>').observations) as obs
                FROM {TARGET_TABLE}
            )
        )
        -- 只取每个指标最新的 5 条记录展示
        SELECT series_id, observation_date, actual_value, ingestion_ts
        FROM UnpackedData
        WHERE row_num <= 5
        ORDER BY series_id, observation_date DESC
    """))
else:
    print(f"⚠️ Table {TARGET_TABLE} does not exist yet. Run the batch first.")
